# 3. Model Train + SHAP / FFA Analysis

**Purpose:** Run pipeline Steps 4–8: model data → PGx → final model → SHAP → FFA/DTW/FP-Growth.

**Flow:** Steps 4 (model data) → 5 (PGx enrichment) → 6 (final model) → 7 (SHAP) → 8 (FFA + DTW + FP-Growth).

**Steps:** Sync inputs → Verify → Step 4 (model data) → Step 5 (PGx) → Step 6 (final model) → Step 7 (SHAP) → Step 8 (FFA).

**Memory:** Pipeline scripts use **DuckDB and Parquet** where possible. Pandas only where required (model/SHAP APIs).

Prerequisites: Cohorts from `1_cohort_workflow.ipynb` (Step 2) and feature importance from `2_feature_importance.ipynb` plus any refined Step 3b feature lists. Run from repo root.

In [5]:
# Setup: paths and project root
import sys
import os
import subprocess
from pathlib import Path

PROJECT_ROOT = Path().resolve()
sys.path.insert(0, str(PROJECT_ROOT))

from py_helpers.env_utils import get_data_root
from py_helpers.workflow_sync_checkpoint import sync_s3_to_local, check_step_checkpoint_exists, save_step_checkpoint

S3_BUCKET = os.environ.get("CPIC_S3_BUCKET", "pgxdatalake")
DATA_ROOT = get_data_root()
AWS_PROFILE = os.environ.get("AWS_PROFILE")

print("CPIC Time-to-Event Analysis - Model + SHAP/FFA Workflow")
print("=" * 60)
print(f"Project root: {PROJECT_ROOT}")
print(f"Data root (NVMe/local): {DATA_ROOT}")
print("=" * 60)

CPIC Time-to-Event Analysis - Model + SHAP/FFA Workflow
Project root: /home/pgx3874/cpic_time_to_event_analysis
Data root (NVMe/local): /mnt/nvme


In [6]:
# Both cohorts use the same production age-band set: 65-74 and 75-84.
from py_helpers.constants import REQUIRED_COHORTS

# Input dirs
COHORTS_ROOT = DATA_ROOT / "gold" / "cohorts"
FI_ROOT = DATA_ROOT / "gold" / "feature_importance"
STEP3_OUTPUTS = STEP3B_OUTPUTS = FI_ROOT

from py_helpers.env_utils import get_model_data_root
MODEL_DATA_ROOT = get_model_data_root()

# Output dirs
FINAL_MODEL_OUTPUTS = PROJECT_ROOT / "6_final_model" / "outputs"
FINAL_MODEL_OUTPUTS_ALT = DATA_ROOT / "6_final_model" / "outputs"
FINAL_MODEL_GOLD = DATA_ROOT / "gold" / "final_model"

print("Cohorts and age bands:")
for cohort, bands in REQUIRED_COHORTS.items():
    print(f"  {cohort}: {bands}")
print("\nInput dirs (for Steps 4-6):")
print(f"  Cohorts (2):               {COHORTS_ROOT}")
print(f"  Feature importance (3/3b): {FI_ROOT}")
print(f"  Model data (4; in/out):    {MODEL_DATA_ROOT}")
print("\nOutput dirs (Step 6):")
print(f"  Project:   {FINAL_MODEL_OUTPUTS}")
print(f"  NVMe:      {FINAL_MODEL_OUTPUTS_ALT}")
print(f"  gold/NVMe: {FINAL_MODEL_GOLD}")

Cohorts and age bands:
  falls: ['65-74', '75-84']
  ed: ['65-74', '75-84']

Input dirs (for Steps 4-6):
  Cohorts (2):               /mnt/nvme/gold/cohorts
  Feature importance (3/3b): /mnt/nvme/gold/feature_importance
  Model data (4; in/out):    /mnt/nvme/cpic_time_to_event/4_model_data

Output dirs (Step 6):
  Project:   /home/pgx3874/cpic_time_to_event_analysis/6_final_model/outputs
  NVMe:      /mnt/nvme/6_final_model/outputs
  gold/NVMe: /mnt/nvme/gold/final_model


## Clear all checkpoints and pipeline outputs (optional - for a fresh run)

Run this cell **once** when you want to rebuild the full pipeline from Step 4 through SHAP/FFA. It (1) clears S3 checkpoints, (2) deletes S3 pipeline outputs so Steps 4 and 6 re-run instead of re-downloading, (3) removes local output directories.

**Safety confirmation required:** edit `CONFIRM_CLEAR_ALL` in the next cell to exactly `CLEAR STEP 4-8 OUTPUTS` before running. Leave it blank to skip. After this, run the Sync cell and then Steps 4 --> 5 --> 6 --> Step 7 --> Step 8.

In [ ]:
# Clear S3 checkpoints, S3 pipeline outputs, and local outputs for a fresh model + SHAP/FFA run.
# Safety: this cell deletes generated project artifacts. It will not run unless
# CONFIRM_CLEAR_ALL exactly matches REQUIRED_CONFIRMATION.
import logging
import shutil
import subprocess
from pathlib import Path

from py_helpers.workflow_sync_checkpoint import clear_step_checkpoints, delete_step_checkpoint

REQUIRED_CONFIRMATION = "CLEAR STEP 4-8 OUTPUTS"
CONFIRM_CLEAR_ALL = ""  # Type CLEAR STEP 4-8 OUTPUTS here to enable deletion.

if CONFIRM_CLEAR_ALL != REQUIRED_CONFIRMATION:
    print("Clear-all skipped.")
    print(f"To run, set CONFIRM_CLEAR_ALL = {REQUIRED_CONFIRMATION!r} and rerun this cell.")
else:
    logger = logging.getLogger(__name__)
    print("Confirmation accepted. Clearing generated Step 4-8 checkpoints and outputs...")

    # 1) S3 checkpoint metadata so steps don't think they're done
    for step in ("4_model_data", "6_final_model"):
        for cohort, bands in REQUIRED_COHORTS.items():
            n = clear_step_checkpoints(step, cohort, bands, logger=None)
            print(f"Cleared {n} checkpoint(s) for {step} / {cohort}")

    # 2) S3 pipeline outputs so Step 4 and Step 6 re-run instead of re-downloading
    _aws = shutil.which("aws") or "aws"
    _profile = ["--profile", AWS_PROFILE] if AWS_PROFILE else []
    for prefix in ("gold/cohorts_model_data/", "gold/final_model/"):
        uri = f"s3://{S3_BUCKET}/{prefix}"
        r = subprocess.run([_aws, "s3", "rm", uri, "--recursive"] + _profile, capture_output=True, text=True)
        if r.returncode == 0:
            print(f"Cleared S3 {uri}")
        else:
            print(f"S3 rm {uri}: exit {r.returncode}; {r.stderr or r.stdout or ''}")

    # 3) Local output directories
    dirs_to_clear = [
        MODEL_DATA_ROOT,
        FINAL_MODEL_OUTPUTS,
        FINAL_MODEL_OUTPUTS_ALT,
        PROJECT_ROOT / "7_shap_analysis" / "outputs",
        PROJECT_ROOT / "8_ffa_analysis" / "outputs",
    ]
    for d in dirs_to_clear:
        d = Path(d)
        if d.exists():
            shutil.rmtree(d)
            print(f"Removed {d}")
        else:
            print(f"(skip, not present) {d}")
    print("Done. Re-run Sync and then Steps 4 --> 5 --> 6 --> Step 7 --> Step 8 for a fresh run.")

## Sync required inputs from S3 to NVMe (idempotent)

Sync **cohorts** (Step 2), **feature importance** (Step 3/3b), and **Step 6** final model outputs from S3 so pipeline and data preparation can read from local/NVMe. **Idempotent:** `aws s3 sync` only updates changed or missing files.

In [ ]:
# Sync cohorts (Step 2), Step 3a/3b feature importance, and Step 6 final models from S3 to DATA_ROOT.
COHORTS_ROOT.mkdir(parents=True, exist_ok=True)
FI_SYNC_TARGET = DATA_ROOT / "gold" / "feature_importance"
FI_SYNC_TARGET.mkdir(parents=True, exist_ok=True)
FINAL_MODEL_GOLD.mkdir(parents=True, exist_ok=True)

sync_s3_to_local(f"s3://{S3_BUCKET}/gold/cohorts/", COHORTS_ROOT, profile=AWS_PROFILE)
sync_s3_to_local(f"s3://{S3_BUCKET}/gold/feature_importance/", FI_SYNC_TARGET, profile=AWS_PROFILE)
sync_s3_to_local(f"s3://{S3_BUCKET}/gold/final_model/", FINAL_MODEL_GOLD, profile=AWS_PROFILE)
print("Sync complete. Run Step 0 verification below.")

## Step 0: Verify inputs (FI required; 4_model_data and Step 6 informational)

**Required:** **Feature importance** (Step 3/3b) — must exist for each cohort/age_band so Pipeline Step 4 can run.

**Informational:** **ModelData** checks `DATA_ROOT/4_model_data` and `PROJECT_ROOT/4_model_data` (same location `create_model_data.py` writes to). **Model** = Step 6 outputs. Both are produced by Pipeline Step 4–6 cells below; if already present, you can skip those cells.

In [ ]:
def check_feature_importance(cohort: str, age_band: str) -> bool:
    """Check if feature importance exists using FileResolver pattern."""
    from py_helpers.file_resolver import FileResolver
    # Check Step 3b refined cohort feature importance first
    resolver_3b = FileResolver(
        file_type="cohort_feature_importance",
        project_root=PROJECT_ROOT,
        cohort=cohort,
        age_band=age_band,
        auto_download=False
    )
    if resolver_3b.exists():
        return True
    # Fallback to Step 3a aggregated feature importance
    resolver_3a = FileResolver(
        file_type="aggregated_feature_importance",
        project_root=PROJECT_ROOT,
        cohort=cohort,
        age_band=age_band,
        auto_download=False
    )
    return resolver_3a.exists()

def check_cohorts(cohort: str, age_band: str) -> bool:
    """Check Step 2 cohort.parquet exists for at least one year (2016-2019). Layout: COHORTS_ROOT/cohort_name=X/event_year=Y/age_band=Z/cohort.parquet."""
    for year in (2016, 2017, 2018, 2019):
        p = COHORTS_ROOT / f"cohort_name={cohort}" / f"event_year={year}" / f"age_band={age_band}" / "cohort.parquet"
        if p.exists():
            return True
    return False

def check_model_data(cohort: str, age_band: str) -> bool:
    """Check model_events.parquet at canonical MODEL_DATA_ROOT (same location create_model_data.py writes to)."""
    p = MODEL_DATA_ROOT / f"cohort_name={cohort}" / f"age_band={age_band}" / "model_events.parquet"
    return p.exists()

def check_final_model(cohort: str, age_band: str) -> bool:
    ab = age_band.replace("-", "_")
    # 1) Project or DATA_ROOT/6_final_model/outputs: cohort/65_74/models/*.joblib
    for base in (FINAL_MODEL_OUTPUTS, FINAL_MODEL_OUTPUTS_ALT):
        model_dir = base / cohort / ab
        if not model_dir.exists():
            continue
        models_sub = model_dir / "models"
        if models_sub.exists() and any(models_sub.glob("*.joblib")):
            return True
        if (model_dir / "feature_schema.json").exists():
            return True
    # 2) DATA_ROOT/gold/final_model (S3-synced): cohort/65-74/*.joblib (hyphen in age_band)
    gold_dir = FINAL_MODEL_GOLD / cohort / age_band
    if gold_dir.exists() and any(gold_dir.glob("*.joblib")):
        return True
    return False

print("Step 0: Verify feature importance (required); cohorts and 4_model_data (Step 4 inputs); Step 6 (informational)")
print("  Locations: Cohorts=COHORTS_ROOT, FI=Step 3/3b, ModelData=MODEL_DATA_ROOT, Model=Step 6 outputs")
fi_ok_all = True
for cohort, bands in REQUIRED_COHORTS.items():
    for age_band in bands:
        cohorts_ok = check_cohorts(cohort, age_band)
        fi_ok = check_feature_importance(cohort, age_band)
        model_data_ok = check_model_data(cohort, age_band)
        model_ok = check_final_model(cohort, age_band)
        if not fi_ok:
            fi_ok_all = False
        status = "ready" if fi_ok else "missing FI"
        print(f"  {cohort} / {age_band}:  Cohorts={cohorts_ok}, FI={fi_ok}, ModelData={model_data_ok}, Model={model_ok}  -> {status}")
if fi_ok_all:
    print("\nAll prerequisites are available to build model data. Run Pipeline Step 4-6 cells below.")
    print("  (If Step 6 is already built elsewhere, you can sync from S3 or skip those cells.)")
else:
    print("\nMissing feature importance for some cohort/age_band. Sync from S3 or run Step 3/3b first, then re-run this cell.")
if fi_ok_all:
    cohorts_missing = [(c, ab) for c, bands in REQUIRED_COHORTS.items() for ab in bands if not check_cohorts(c, ab)]
    if cohorts_missing:
        print("\nCohorts=False for some cohort/age_band. Sync gold/cohorts from S3 (run Sync cell) or run Step 2. Expected layout: COHORTS_ROOT/cohort_name=X/event_year=Y/age_band=Z/cohort.parquet (Y in 2016-2019).")


# Pipeline Phase 4: Model data

Build `model_events.parquet` for each cohort/age_band from Step 2 cohort data and Step 3b feature importance. Outputs go to `MODEL_DATA_ROOT/cohort_name={cohort}/age_band={age_band}/model_events.parquet`. Run the cell below for all PGx cohorts/age_bands defined in this notebook.

In [ ]:
# Pipeline Step 4: BUILD model_events.parquet by running create_model_data.py, then QA.
# The script READS: COHORTS_ROOT (cohort.parquet), gold/medical, gold/pharmacy, and feature importance.
# It WRITES: MODEL_DATA_ROOT/cohort_name={cohort}/age_band={age_band}/model_events.parquet
import duckdb

FORCE_STEP4_ED = True
FORCE_STEP4_ALL = False
REBUILT_STEP4 = set()

def _model_data_candidates(cohort: str, age_band: str):
    """Canonical location for model_events.parquet (Step 4 writes to MODEL_DATA_ROOT)."""
    return [MODEL_DATA_ROOT]

def _model_data_path(cohort: str, age_band: str) -> Path:
    """Resolve model_events.parquet path (Step 4 writes to get_model_data_root() = DATA_ROOT or PROJECT on Linux)."""
    for base in _model_data_candidates(cohort, age_band):
        p = base / f"cohort_name={cohort}" / f"age_band={age_band}" / "model_events.parquet"
        if p.exists():
            return p
    return None

def _log_model_data_qa(cohort: str, age_band: str) -> None:
    """Log location, target distribution, control:case ratio, and patient event-count balance."""
    path = _model_data_path(cohort, age_band)
    if not path:
        print(f"  [WARN] model_events.parquet not found for {cohort}/{age_band}")
        for base in _model_data_candidates(cohort, age_band):
            p = base / f"cohort_name={cohort}" / f"age_band={age_band}" / "model_events.parquet"
            print(f"    Checked: {p}  (exists: {p.exists()})")
        print(f"    Build did not write output. Check script stdout above: [INFO] data roots and example cohort path (exists=?). Layout must be {COHORTS_ROOT}/cohort_name=X/event_year=Y/age_band=Z/cohort.parquet (Y in 2016-2019). Sync cohorts to COHORTS_ROOT if needed, then re-run this cell.")
        return
    print(f"  Location: {path}")
    con = duckdb.connect()
    try:
        dist = con.execute("SELECT target, COUNT(*)::BIGINT AS n FROM read_parquet(?) GROUP BY target ORDER BY target", [str(path)]).fetchall()
        total = sum(row[1] for row in dist)
        by_target = {int(row[0]): int(row[1]) for row in dist}
        n_controls = by_target.get(0, 0)
        n_cases = by_target.get(1, 0)
        ratio = (n_controls / n_cases) if n_cases else 0
        print(f"  Target distribution: {by_target} (total rows: {total:,})")
        print(f"  Control:case ratio: {n_controls:,}:{n_cases:,} = {ratio:.2f}:1")
        event_counts = con.execute(
            """
            SELECT
              target,
              COUNT(DISTINCT mi_person_key)::BIGINT AS patients,
              AVG(n_events) AS mean_events,
              MEDIAN(n_events) AS median_events,
              MIN(n_events) AS min_events,
              MAX(n_events) AS max_events
            FROM (
              SELECT mi_person_key, target, COUNT(*) AS n_events
              FROM read_parquet(?)
              GROUP BY mi_person_key, target
            )
            GROUP BY target
            ORDER BY target
            """,
            [str(path)],
        ).df()
        print("  Patient-level event-count QA:")
        print(event_counts.to_string(index=False))
    finally:
        con.close()

for cohort, bands in REQUIRED_COHORTS.items():
    for age_band in bands:
        print(f"--> Step 4: {cohort} / {age_band} (building model_events.parquet)")
        force_step4 = FORCE_STEP4_ALL or (FORCE_STEP4_ED and cohort == "ed")
        existed_before = _model_data_path(cohort, age_band) is not None
        cmd = [sys.executable, "create_model_data.py", "--cohort", cohort, "--age_band", age_band]
        if force_step4:
            cmd.append("--force-rebuild")
        r = subprocess.run(
            cmd,
            cwd=PROJECT_ROOT / "4_model_data",
            capture_output=False,
        )
        if r.returncode != 0:
            raise SystemExit(r.returncode)
        if force_step4 or not existed_before:
            REBUILT_STEP4.add((cohort, age_band))
        _log_model_data_qa(cohort, age_band)
print("Step 4 complete.")

# Pipeline Phase 5: PGx analysis

Add PGx features (e.g. CPIC drug counts) to model data. Reads from Step 4 outputs and writes updated model data used by Step 6. Run for each cohort/age_band.

In [ ]:
# Pipeline Step 5: run_analysis.py for each REQUIRED_COHORTS (cohort, age_band)
# Set FORCE_STEP5 = True to re-run even when S3 outputs or checkpoints exist
FORCE_STEP5 = True
for cohort, bands in REQUIRED_COHORTS.items():
    for age_band in bands:
        print(f"--> Step 5: {cohort} / {age_band}")
        cmd = [sys.executable, "run_analysis.py", "--cohort-name", cohort, "--age-band", age_band]
        if FORCE_STEP5:
            cmd.append("--force")
        r = subprocess.run(cmd, cwd=PROJECT_ROOT / "5_pgx_analysis")
        if r.returncode != 0:
            raise SystemExit(r.returncode)
print("Step 5 complete.")

# Pre-training QA: model events

Run this gate after Step 4 model-data creation and Step 5 PGx enrichment, before Step 6 final model training. It validates the local `model_events.parquet` files that Step 6 will read and fails fast if any cohort/age-band is missing controls, target labels, target dates for cases, or has case events on/after the target date.

The Athena QA in `aws/athena/` gives the same style of S3-backed coverage check for persisted outputs; this cell is the local pre-training gate.

In [ ]:
# Pre-training QA gate for Step 4 model_events.parquet.
# This must pass before Step 6 final model training.
import duckdb
import pandas as pd

PRETRAIN_QA_PASSED = False
PRETRAIN_QA_ROWS = []
PRETRAIN_QA_FAILURES = []


def _pretrain_model_events_path(cohort: str, age_band: str) -> Path:
    return MODEL_DATA_ROOT / f"cohort_name={cohort}" / f"age_band={age_band}" / "model_events.parquet"


def _target_date_col(cohort: str) -> str:
    return "first_fall_date" if cohort == "falls" else "first_ed_date"


def _qa_one_model_events(cohort: str, age_band: str) -> None:
    path = _pretrain_model_events_path(cohort, age_band)
    if not path.exists():
        PRETRAIN_QA_FAILURES.append(f"{cohort}/{age_band}: missing model_events.parquet at {path}")
        PRETRAIN_QA_ROWS.append({"cohort": cohort, "age_band": age_band, "status": "missing", "path": str(path)})
        return

    target_date_col = _target_date_col(cohort)
    path_sql = str(path).replace("'", "''")
    con = duckdb.connect()
    try:
        cols = [row[0] for row in con.execute(f"DESCRIBE SELECT * FROM read_parquet('{path_sql}')").fetchall()]
        required_cols = {"mi_person_key", "event_date", "target", target_date_col}
        missing_cols = sorted(required_cols - set(cols))
        if missing_cols:
            PRETRAIN_QA_FAILURES.append(f"{cohort}/{age_band}: missing required columns {missing_cols} in {path}")
            PRETRAIN_QA_ROWS.append({"cohort": cohort, "age_band": age_band, "status": "missing_columns", "path": str(path)})
            return

        qa = con.execute(
            f"""
            SELECT
                COUNT(*)::BIGINT AS rows,
                COUNT(DISTINCT mi_person_key)::BIGINT AS patients,
                COUNT(*) FILTER (WHERE target = 1)::BIGINT AS case_rows,
                COUNT(*) FILTER (WHERE target = 0)::BIGINT AS control_rows,
                COUNT(DISTINCT CASE WHEN target = 1 THEN mi_person_key END)::BIGINT AS case_patients,
                COUNT(DISTINCT CASE WHEN target = 0 THEN mi_person_key END)::BIGINT AS control_patients,
                SUM(CASE WHEN mi_person_key IS NULL THEN 1 ELSE 0 END)::BIGINT AS null_patient_rows,
                SUM(CASE WHEN event_date IS NULL THEN 1 ELSE 0 END)::BIGINT AS null_event_date_rows,
                SUM(CASE WHEN target NOT IN (0, 1) OR target IS NULL THEN 1 ELSE 0 END)::BIGINT AS invalid_target_rows,
                SUM(CASE WHEN target = 1 AND "{target_date_col}" IS NULL THEN 1 ELSE 0 END)::BIGINT AS case_rows_missing_target_date,
                SUM(
                    CASE
                        WHEN target = 1
                         AND "{target_date_col}" IS NOT NULL
                         AND event_date IS NOT NULL
                         AND CAST(event_date AS DATE) >= CAST("{target_date_col}" AS DATE)
                        THEN 1 ELSE 0
                    END
                )::BIGINT AS case_rows_on_or_after_target,
                MIN(event_date) AS min_event_date,
                MAX(event_date) AS max_event_date
            FROM read_parquet('{path_sql}')
            """
        ).df().iloc[0].to_dict()

        row = {
            "cohort": cohort,
            "age_band": age_band,
            "status": "ok",
            "path": str(path),
            **qa,
        }
        row["control_to_case_patient_ratio"] = (
            float(row["control_patients"]) / float(row["case_patients"])
            if int(row["case_patients"] or 0) else None
        )

        if "drug_name" in cols:
            drug_qa = con.execute(
                f"""
                SELECT
                    COUNT(*) FILTER (WHERE drug_name IS NOT NULL)::BIGINT AS drug_rows,
                    COUNT(*) FILTER (WHERE target = 1 AND drug_name IS NOT NULL)::BIGINT AS case_drug_rows,
                    COUNT(*) FILTER (WHERE target = 0 AND drug_name IS NOT NULL)::BIGINT AS control_drug_rows
                FROM read_parquet('{path_sql}')
                """
            ).df().iloc[0].to_dict()
            row.update(drug_qa)
        else:
            row.update({"drug_rows": None, "case_drug_rows": None, "control_drug_rows": None})

        PRETRAIN_QA_ROWS.append(row)

        hard_fail_checks = {
            "no_rows": int(row["rows"] or 0) == 0,
            "no_case_rows": int(row["case_rows"] or 0) == 0,
            "no_control_rows": int(row["control_rows"] or 0) == 0,
            "no_case_patients": int(row["case_patients"] or 0) == 0,
            "no_control_patients": int(row["control_patients"] or 0) == 0,
            "null_patient_rows": int(row["null_patient_rows"] or 0) > 0,
            "null_event_date_rows": int(row["null_event_date_rows"] or 0) > 0,
            "invalid_target_rows": int(row["invalid_target_rows"] or 0) > 0,
            "case_rows_missing_target_date": int(row["case_rows_missing_target_date"] or 0) > 0,
            "case_rows_on_or_after_target": int(row["case_rows_on_or_after_target"] or 0) > 0,
        }
        if cohort == "ed":
            hard_fail_checks["ed_no_case_drug_rows"] = int(row.get("case_drug_rows") or 0) == 0
            hard_fail_checks["ed_no_control_drug_rows"] = int(row.get("control_drug_rows") or 0) == 0

        failed = [name for name, failed_check in hard_fail_checks.items() if failed_check]
        if failed:
            PRETRAIN_QA_FAILURES.append(f"{cohort}/{age_band}: failed {failed}; path={path}")
    finally:
        con.close()


for cohort, bands in REQUIRED_COHORTS.items():
    for age_band in bands:
        print(f"--> Pre-training QA: {cohort} / {age_band}")
        _qa_one_model_events(cohort, age_band)

qa_df = pd.DataFrame(PRETRAIN_QA_ROWS)
if not qa_df.empty:
    display_cols = [
        "cohort",
        "age_band",
        "status",
        "rows",
        "patients",
        "case_rows",
        "control_rows",
        "case_patients",
        "control_patients",
        "control_to_case_patient_ratio",
        "null_event_date_rows",
        "case_rows_missing_target_date",
        "case_rows_on_or_after_target",
        "drug_rows",
        "case_drug_rows",
        "control_drug_rows",
    ]
    display(qa_df[[c for c in display_cols if c in qa_df.columns]])

if PRETRAIN_QA_FAILURES:
    print("[X] Pre-training QA failed:")
    for failure in PRETRAIN_QA_FAILURES:
        print(f"  - {failure}")
    print("\n[ACTION] Pull the latest repo updates on EC2, rerun the Step 4 model-data cell, then rerun this QA cell.")
    print("         Step 4 now rebuilds cached model_events files with null event_date rows.")
    raise RuntimeError("Pre-training model-events QA failed; fix Step 4 model_events before running Step 6.")

PRETRAIN_QA_PASSED = True
print("[1] Pre-training model-events QA passed for all configured cohorts and age bands.")

# Pipeline Phase 6: Model Outputs, SHAP, FFA, and Performance Review

Train final models per cohort/age band, then run the interpretability and review steps in this order:

- **SHAP Values**
- **FFA Analysis**
- **Combined SHAP+FFA**
  - **Top 20 Consensus Feature Importance**
- **Model Performance per density bin**
  - **Model Performance Summary**

Step 6 model training uses **per-bin** training by default (`--train-mode per_bin`) for both `falls` and `ed`: separate models under `outputs/.../bin_models/{low|medium|high|extreme}/`, then mirrors one bin (prefer `medium`) to the cohort-level `outputs/.../{age_band}/` tree for prepare_models / Lambda. Use `--train-mode aggregate` for a single cohort-wide model only, or `both` for cohort-wide plus per-bin.

In [7]:
# Pipeline Step 6: run_final_model.py for each REQUIRED_COHORTS (cohort, age_band)
# Note: script uses --age_band (underscore). Default --train-mode is per_bin (omit flag).
if globals().get("PRETRAIN_QA_PASSED") is not True:
    raise RuntimeError("Run and pass the Pre-training QA: model events cell before Step 6 training.")

FORCE_STEP6 = False
FORCE_STEP6_REBUILT_ONLY = True
STEP6_TRAIN_MODE = None
rebuilt_step4 = globals().get("REBUILT_STEP4", set())
for cohort, bands in REQUIRED_COHORTS.items():
    for age_band in bands:
        print(f"--> Step 6: {cohort} / {age_band}")
        cmd = [sys.executable, "run_final_model.py", "--cohort", cohort, "--age_band", age_band]
        if STEP6_TRAIN_MODE:
            cmd.extend(["--train-mode", STEP6_TRAIN_MODE])
        force_step6 = FORCE_STEP6 or (FORCE_STEP6_REBUILT_ONLY and (cohort, age_band) in rebuilt_step4)
        if force_step6:
            cmd.append("--force-retrain")
        r = subprocess.run(
            cmd,
            cwd=PROJECT_ROOT / "6_final_model",
        )
        if r.returncode != 0:
            cmd_text = " ".join(str(part) for part in cmd)
            raise RuntimeError(
                f"Step 6 failed for {cohort}/{age_band} with return code {r.returncode}. "
                f"Command: {cmd_text}"
            )
print("Step 6 complete.")

--> Step 6: falls / 65-74
--> Step 6: falls / 75-84
--> Step 6: ed / 65-74
--> Step 6: ed / 75-84
Step 6 complete.


## SHAP Values

**7_shap_analysis/run_shap_analysis.py** runs per-bin SHAP for bins trained in Step 6 and falls back to cohort-level SHAP if only aggregate models exist.

In [8]:
import logging
logger = logging.getLogger(__name__)

from py_helpers.event_density_utils import (
    cohort_aggregate_final_model_has_artifacts,
    list_trained_density_bins,
)

SHAP_SCRIPT = PROJECT_ROOT / "7_shap_analysis" / "run_shap_analysis.py"
for cohort, bands in REQUIRED_COHORTS.items():
    for age_band in bands:
        trained = list_trained_density_bins(PROJECT_ROOT, cohort, age_band)
        if trained:
            for bin_name in trained:
                print(f"--> Step 7 (SHAP bin={bin_name}): {cohort} / {age_band}")
                r = subprocess.run(
                    [sys.executable, str(SHAP_SCRIPT),
                     "--cohort", cohort, "--age_band", age_band, "--bin", bin_name],
                    cwd=PROJECT_ROOT,
                )
                if r.returncode != 0:
                    raise SystemExit(r.returncode)
        elif cohort_aggregate_final_model_has_artifacts(PROJECT_ROOT, cohort, age_band):
            print(f"--> Step 7 (SHAP cohort-level): {cohort} / {age_band}")
            r = subprocess.run(
                [sys.executable, str(SHAP_SCRIPT), "--cohort", cohort, "--age_band", age_band],
                cwd=PROJECT_ROOT,
            )
            if r.returncode != 0:
                raise SystemExit(r.returncode)
        else:
            print(f"[skip] Step 7: no Step 6 models for {cohort} / {age_band}")
print("Step 7 (SHAP) complete.")

--> Step 7 (SHAP bin=low): falls / 65-74

=== Per-bin Step 6 output validation: falls / 65-74 ===
  Canonical path: 6_final_model/outputs/falls/65_74/bin_models/<bin>/
  [OK     ] low       /home/pgx3874/cpic_time_to_event_analysis/6_final_model/outputs/falls/65_74/bin_models/low
  Summary - found: ['low']  |  missing: (none)

[SKIP] Step 7 outputs already exist locally for falls/65-74
--> Step 7 (SHAP bin=medium): falls / 65-74

=== Per-bin Step 6 output validation: falls / 65-74 ===
  Canonical path: 6_final_model/outputs/falls/65_74/bin_models/<bin>/
  [OK     ] medium    /home/pgx3874/cpic_time_to_event_analysis/6_final_model/outputs/falls/65_74/bin_models/medium
  Summary - found: ['medium']  |  missing: (none)

[SKIP] Step 7 outputs already exist locally for falls/65-74
--> Step 7 (SHAP bin=high): falls / 65-74

=== Per-bin Step 6 output validation: falls / 65-74 ===
  Canonical path: 6_final_model/outputs/falls/65_74/bin_models/<bin>/
  [OK     ] high      /home/pgx3874/cpic_tim

### Review: SHAP Output Files

Confirm SHAP value parquets exist for each cohort/age band or per-bin model immediately after SHAP completes.

In [9]:
SHAP_OUTPUTS = PROJECT_ROOT / "7_shap_analysis" / "outputs"
print("SHAP outputs:")
if SHAP_OUTPUTS.exists():
    files = sorted(SHAP_OUTPUTS.rglob("*.parquet"))
    for f in files:
        size_mb = f.stat().st_size / 1e6
        print(f"  {f.relative_to(PROJECT_ROOT)}  ({size_mb:.1f} MB)")
    if not files:
        print("  (no parquet files yet - run Step 7)")
else:
    print(f"  {SHAP_OUTPUTS} does not exist - run Step 7")

SHAP outputs:
  7_shap_analysis/outputs/ed/65_74/bin_models/extreme/ed_65_74_shap_sample_values_catboost.parquet  (0.1 MB)
  7_shap_analysis/outputs/ed/65_74/bin_models/extreme/ed_65_74_shap_sample_values_xgboost.parquet  (0.0 MB)
  7_shap_analysis/outputs/ed/65_74/bin_models/high/ed_65_74_shap_sample_values_catboost.parquet  (0.4 MB)
  7_shap_analysis/outputs/ed/65_74/bin_models/high/ed_65_74_shap_sample_values_xgboost.parquet  (0.2 MB)
  7_shap_analysis/outputs/ed/65_74/bin_models/low/ed_65_74_shap_sample_values_catboost.parquet  (0.3 MB)
  7_shap_analysis/outputs/ed/65_74/bin_models/low/ed_65_74_shap_sample_values_xgboost.parquet  (0.0 MB)
  7_shap_analysis/outputs/ed/65_74/bin_models/medium/ed_65_74_shap_sample_values_catboost.parquet  (0.2 MB)
  7_shap_analysis/outputs/ed/65_74/bin_models/medium/ed_65_74_shap_sample_values_xgboost.parquet  (0.1 MB)
  7_shap_analysis/outputs/ed/75_84/bin_models/extreme/ed_75_84_shap_sample_values_catboost.parquet  (0.0 MB)
  7_shap_analysis/outputs

## FFA Analysis

**8_ffa_analysis/run_ffa_workflow.py** runs after SHAP. Per-bin resolution mirrors the SHAP step, writes FFA outputs under `8_ffa_analysis/outputs/`, and then invokes the combined SHAP+FFA artifact builder.

In [10]:
from py_helpers.event_density_utils import (
    cohort_aggregate_final_model_has_artifacts,
    list_trained_density_bins,
)

FFA_SCRIPT = PROJECT_ROOT / "8_ffa_analysis" / "run_ffa_workflow.py"
for cohort, bands in REQUIRED_COHORTS.items():
    for age_band in bands:
        trained = list_trained_density_bins(PROJECT_ROOT, cohort, age_band)
        if trained:
            for bin_name in trained:
                print(f"--> Step 8 (FFA + combine bin={bin_name}): {cohort} / {age_band}")
                cmd = [
                    sys.executable, str(FFA_SCRIPT),
                    "--cohort", cohort, "--age-band", age_band, "--bin", bin_name,
                ]
                r = subprocess.run(cmd, cwd=PROJECT_ROOT)
                if r.returncode != 0:
                    cmd_text = " ".join(str(part) for part in cmd)
                    raise RuntimeError(
                        f"Step 8 FFA+combine failed for {cohort}/{age_band} bin={bin_name} "
                        f"with return code {r.returncode}. Command: {cmd_text}"
                    )
        elif cohort_aggregate_final_model_has_artifacts(PROJECT_ROOT, cohort, age_band):
            print(f"--> Step 8 (FFA + combine cohort-level): {cohort} / {age_band}")
            cmd = [sys.executable, str(FFA_SCRIPT), "--cohort", cohort, "--age-band", age_band]
            r = subprocess.run(cmd, cwd=PROJECT_ROOT)
            if r.returncode != 0:
                cmd_text = " ".join(str(part) for part in cmd)
                raise RuntimeError(
                    f"Step 8 FFA+combine failed for {cohort}/{age_band} "
                    f"with return code {r.returncode}. Command: {cmd_text}"
                )
        else:
            print(f"[skip] Step 8: no Step 6 models for {cohort} / {age_band}")
print("Step 8 (FFA + combined SHAP/FFA artifacts) complete.")

--> Step 8 (FFA + combine bin=low): falls / 65-74


2026-06-21 02:27:23,636 - INFO - FFA outputs already exist locally; skipping FFA computation.
2026-06-21 02:27:24,430 - INFO - Found credentials from IAM Role: EC2_Spot
2026-06-21 02:27:24,508 - INFO - [OK] File already exists in S3: s3://pgxdatalake/gold/cpic_time_to_event/ffa_analysis/falls/65-74/bin_models/low/xgboost/axp_explanations.parquet (skipping upload)
2026-06-21 02:27:24,522 - INFO - [OK] File already exists in S3: s3://pgxdatalake/gold/cpic_time_to_event/ffa_analysis/falls/65-74/bin_models/low/xgboost/feature_importance_axp.parquet (skipping upload)
2026-06-21 02:27:24,535 - INFO - [OK] File already exists in S3: s3://pgxdatalake/gold/cpic_time_to_event/ffa_analysis/falls/65-74/bin_models/low/ffa_causal_factors.csv (skipping upload)
2026-06-21 02:27:24,568 - INFO - Running combine step: /home/pgx3874/jupyter-env/bin/python3.11 /home/pgx3874/cpic_time_to_event_analysis/10_analysis_results/data_preparation/combine_shap_ffa_results.py --cohort falls --age-band 65-74 --workers


SHAP + FFA COMBINED ANALYSIS SUMMARY
  Cohort: falls / 65-74

FEATURE TYPES (combined importance):
  drug: 11, icd: 23, cpt: 67, other: 1

CONSENSUS FEATURES:
  - Consensus features: 17
  - SHAP-only features: 3
  - FFA-only features: 3
  - Consensus rate: 85.0%

COMBINED FEATURE IMPORTANCE (Top 10):
  1. pgx_num_drugs: 1.0000 (SHAP: 1.000, FFA: 1.000)
  2. item_cpt_99213: 0.4944 (SHAP: 0.702, FFA: 0.287)
  3. item_cpt_99214: 0.4047 (SHAP: 0.584, FFA: 0.226)
  4. item_drug_OMEPRAZOLE: 0.1511 (SHAP: 0.260, FFA: 0.042)
  5. item_drug_SIMVASTATIN: 0.1396 (SHAP: 0.241, FFA: 0.039)
  6. item_cpt_99212: 0.1065 (SHAP: 0.190, FFA: 0.023)
  7. item_cpt_85025: 0.1054 (SHAP: 0.183, FFA: 0.027)
  8. item_cpt_84443: 0.1033 (SHAP: 0.186, FFA: 0.020)
  9. item_drug_LISINOPRIL: 0.1009 (SHAP: 0.176, FFA: 0.025)
  10. item_cpt_90662: 0.0893 (SHAP: 0.154, FFA: 0.024)

PATIENT EXPLANATIONS:
  - Total patients analyzed: 377
  - Patients with consensus features: 377


=== Per-bin Step 6 output validation: 

2026-06-21 02:27:26,926 - INFO - FFA outputs already exist locally; skipping FFA computation.
2026-06-21 02:27:27,720 - INFO - Found credentials from IAM Role: EC2_Spot
2026-06-21 02:27:27,800 - INFO - [OK] File already exists in S3: s3://pgxdatalake/gold/cpic_time_to_event/ffa_analysis/falls/65-74/bin_models/medium/xgboost/axp_explanations.parquet (skipping upload)
2026-06-21 02:27:27,810 - INFO - [OK] File already exists in S3: s3://pgxdatalake/gold/cpic_time_to_event/ffa_analysis/falls/65-74/bin_models/medium/xgboost/feature_importance_axp.parquet (skipping upload)
2026-06-21 02:27:27,825 - INFO - [OK] File already exists in S3: s3://pgxdatalake/gold/cpic_time_to_event/ffa_analysis/falls/65-74/bin_models/medium/ffa_causal_factors.csv (skipping upload)
2026-06-21 02:27:27,867 - INFO - Running combine step: /home/pgx3874/jupyter-env/bin/python3.11 /home/pgx3874/cpic_time_to_event_analysis/10_analysis_results/data_preparation/combine_shap_ffa_results.py --cohort falls --age-band 65-74 


SHAP + FFA COMBINED ANALYSIS SUMMARY
  Cohort: falls / 65-74

FEATURE TYPES (combined importance):
  drug: 17, icd: 35, cpt: 100, other: 1

CONSENSUS FEATURES:
  - Consensus features: 19
  - SHAP-only features: 1
  - FFA-only features: 1
  - Consensus rate: 95.0%

COMBINED FEATURE IMPORTANCE (Top 10):
  1. pgx_num_drugs: 1.0000 (SHAP: 1.000, FFA: 1.000)
  2. item_cpt_99215: 0.3955 (SHAP: 0.586, FFA: 0.205)
  3. item_cpt_99214: 0.2646 (SHAP: 0.423, FFA: 0.106)
  4. item_cpt_97110: 0.2038 (SHAP: 0.345, FFA: 0.063)
  5. item_icd_Z23: 0.1938 (SHAP: 0.323, FFA: 0.065)
  6. item_cpt_93005: 0.1492 (SHAP: 0.258, FFA: 0.040)
  7. item_cpt_88305: 0.1473 (SHAP: 0.262, FFA: 0.033)
  8. item_cpt_93000: 0.1186 (SHAP: 0.203, FFA: 0.034)
  9. item_cpt_36415: 0.1077 (SHAP: 0.193, FFA: 0.022)
  10. item_cpt_99213: 0.1023 (SHAP: 0.182, FFA: 0.023)

PATIENT EXPLANATIONS:
  - Total patients analyzed: 266
  - Patients with consensus features: 266


=== Per-bin Step 6 output validation: falls / 65-74 ===
  

...


=== Per-bin Step 6 output validation: ed / 75-84 ===
  Canonical path: 6_final_model/outputs/ed/75_84/bin_models/<bin>/
  [OK     ] extreme   /home/pgx3874/cpic_time_to_event_analysis/6_final_model/outputs/ed/75_84/bin_models/extreme
  Summary - found: ['extreme']  |  missing: (none)



Generating explanations:   0%|          | 0/73 [00:00<?, ?it/s]2026-06-21 02:28:27,099 - INFO - Completed 1/73 instances
2026-06-21 02:28:27,113 - INFO - Completed 2/73 instances
2026-06-21 02:28:27,119 - INFO - Completed 3/73 instances
2026-06-21 02:28:27,119 - INFO - Completed 4/73 instances
2026-06-21 02:28:27,120 - INFO - Completed 5/73 instances
Generating explanations: 100%|██████████| 73/73 [00:00<00:00, 131.46it/s]
2026-06-21 02:28:27,736 - INFO - Wrote /home/pgx3874/cpic_time_to_event_analysis/8_ffa_analysis/outputs/ed/75_84/bin_models/extreme/xgboost/axp_explanations.parquet (73 rows)
2026-06-21 02:28:27,741 - INFO - Wrote FFA outputs under /home/pgx3874/cpic_time_to_event_analysis/8_ffa_analysis/outputs/ed/75_84/bin_models/extreme
2026-06-21 02:28:27,825 - INFO - [OK] Uploaded to S3: s3://pgxdatalake/gold/cpic_time_to_event/ffa_analysis/ed/75-84/bin_models/extreme/xgboost/axp_explanations.parquet
2026-06-21 02:28:27,866 - INFO - [OK] Uploaded to S3: s3://pgxdatalake/gold/cpi


SHAP + FFA COMBINED ANALYSIS SUMMARY
  Cohort: ed / 75-84

FEATURE TYPES (combined importance):
  drug: 9, icd: 0, cpt: 0, other: 1

CONSENSUS FEATURES:
  - Consensus features: 10
  - SHAP-only features: 0
  - FFA-only features: 0
  - Consensus rate: 50.0%

COMBINED FEATURE IMPORTANCE (Top 10):
  1. pgx_num_drugs: 1.0000 (SHAP: 1.000, FFA: 1.000)
  2. item_drug_AZITHROMYCIN: 0.1522 (SHAP: 0.269, FFA: 0.035)
  3. item_drug_LISINOPRIL: 0.0751 (SHAP: 0.139, FFA: 0.011)
  4. item_drug_CARVEDILOL: 0.0704 (SHAP: 0.133, FFA: 0.008)
  5. item_drug_SIMVASTATIN: 0.0488 (SHAP: 0.091, FFA: 0.006)
  6. item_drug_OMEPRAZOLE: 0.0360 (SHAP: 0.069, FFA: 0.003)
  7. item_drug_LORAZEPAM: 0.0335 (SHAP: 0.064, FFA: 0.003)
  8. item_drug_CEPHALEXIN: 0.0098 (SHAP: 0.019, FFA: 0.001)
  9. item_drug_HYDROCHLOROTHIAZIDE: 0.0035 (SHAP: 0.007, FFA: 0.000)
  10. item_drug_AMOXICILLIN: 0.0000 (SHAP: 0.000, FFA: 0.000)

PATIENT EXPLANATIONS:
  - Total patients analyzed: 73
  - Patients with consensus features: 73



2026-06-21 02:28:29,743 - INFO - Step 8 FFA + combined SHAP/FFA workflow complete.


Step 8 (FFA + combined SHAP/FFA artifacts) complete.


## Combined SHAP+FFA

The Step 8 workflow also writes combined interaction/scenario artifacts under `10_analysis_results/visualizations/scenario/{cohort}/{age_band}/` and mirrors project-scoped gold artifacts to S3 when configured.

### Top 20 Consensus Feature Importance

The final review and visualization of the top 20 consensus features is handled in `4_results_review.ipynb`, using the combined SHAP+FFA consensus importance artifacts generated here.

In [11]:
import json
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from py_helpers.event_density_utils import (
    cohort_aggregate_final_model_has_artifacts,
    list_trained_density_bins,
)

COMBINED_ROOT = PROJECT_ROOT / "10_analysis_results" / "visualizations" / "scenario"
combined_status_rows = []
combined_frames = []

for cohort, bands in REQUIRED_COHORTS.items():
    for age_band in bands:
        ab_f = age_band.replace("-", "_")
        trained_bins = list_trained_density_bins(PROJECT_ROOT, cohort, age_band)
        bin_names = list(trained_bins)
        if not bin_names and cohort_aggregate_final_model_has_artifacts(PROJECT_ROOT, cohort, age_band):
            bin_names = [None]

        for bin_name in bin_names:
            scenario_dir = COMBINED_ROOT / cohort / ab_f
            label_bin = bin_name or "aggregate"
            if bin_name:
                scenario_dir = scenario_dir / "bin_models" / bin_name

            combined_path = scenario_dir / "combined_importance.csv"
            consensus_path = scenario_dir / "consensus_features.json"
            patient_path = scenario_dir / "patient_explanations.csv"
            dashboard_path = scenario_dir / "dashboard_data.json"

            combined_exists = combined_path.exists()
            consensus_exists = consensus_path.exists()
            combined_status_rows.append({
                "cohort": cohort,
                "age_band": age_band,
                "bin": label_bin,
                "combined_importance": combined_exists,
                "consensus_features": consensus_exists,
                "patient_explanations": patient_path.exists(),
                "dashboard_data": dashboard_path.exists(),
                "scenario_dir": str(scenario_dir.relative_to(PROJECT_ROOT)) if scenario_dir.exists() else str(scenario_dir),
            })

            if not combined_exists:
                continue

            df = pd.read_csv(combined_path)
            if df.empty or "feature" not in df.columns:
                continue

            consensus_features = []
            if consensus_exists:
                with open(consensus_path, "r", encoding="utf-8") as f:
                    consensus_features = json.load(f).get("consensus_features", [])
            if consensus_features:
                df = df[df["feature"].astype(str).isin(set(map(str, consensus_features)))].copy()
            else:
                df = df.copy()
                df["consensus_warning"] = "consensus_features.json missing or empty; showing combined importance"

            df["cohort"] = cohort
            df["age_band"] = age_band
            df["bin"] = label_bin
            combined_frames.append(df)

combined_status = pd.DataFrame(combined_status_rows)
print("Combined SHAP+FFA artifact coverage:")
display(combined_status)

if combined_frames:
    combined_consensus_importance = pd.concat(combined_frames, ignore_index=True)
    combined_consensus_importance = combined_consensus_importance.sort_values(
        ["cohort", "age_band", "bin", "combined_importance", "feature"],
        ascending=[True, True, True, False, True],
    )
    top20_consensus_by_bin = (
        combined_consensus_importance
        .groupby(["cohort", "age_band", "bin"], as_index=False, group_keys=False)
        .head(20)
        .copy()
    )
    print("Top 20 Consensus Feature Importance per cohort / age band / bin:")
    print(f"Rows shown in heatmap input: {len(top20_consensus_by_bin):,}")

    heat = top20_consensus_by_bin.copy()
    heat["panel"] = heat["cohort"] + " / " + heat["age_band"] + " / " + heat["bin"].astype(str)
    feature_order = (
        heat.groupby("feature")["combined_importance"]
        .max()
        .sort_values(ascending=False)
        .index
    )
    mat = (
        heat.pivot_table(
            index="feature",
            columns="panel",
            values="combined_importance",
            aggfunc="max",
            fill_value=0,
        )
        .reindex(feature_order)
    )
    plt.figure(figsize=(max(10, 0.9 * mat.shape[1]), max(8, 0.28 * mat.shape[0])))
    sns.heatmap(mat, cmap="viridis", linewidths=0.2, linecolor="white")
    plt.title("Top 20 Consensus Feature Importance per Cohort / Bin")
    plt.xlabel("Cohort / age band / bin")
    plt.ylabel("Consensus feature")
    plt.xticks(rotation=45, ha="right")
    plt.tight_layout()
    plt.show()
else:
    combined_consensus_importance = pd.DataFrame()
    top20_consensus_by_bin = pd.DataFrame()
    print("No combined SHAP+FFA artifacts found yet. Run the FFA Analysis cell above.")

Combined SHAP+FFA artifact coverage:


,cohort,age_band,bin,combined_importance,consensus_features,patient_explanations,dashboard_data,scenario_dir
0,falls,65-74,low,True,True,True,True,10_analysis_results/visualizations/scenario/fa...
1,falls,65-74,medium,True,True,True,True,10_analysis_results/visualizations/scenario/fa...
2,falls,65-74,high,True,True,True,True,10_analysis_results/visualizations/scenario/fa...
3,falls,65-74,extreme,True,True,True,True,10_analysis_results/visualizations/scenario/fa...
4,falls,75-84,low,True,True,True,True,10_analysis_results/visualizations/scenario/fa...
5,falls,75-84,medium,True,True,True,True,10_analysis_results/visualizations/scenario/fa...
6,falls,75-84,high,True,True,True,True,10_analysis_results/visualizations/scenario/fa...
7,falls,75-84,extreme,True,True,True,True,10_analysis_results/visualizations/scenario/fa...
8,ed,65-74,low,True,True,True,True,10_analysis_results/visualizations/scenario/ed...
9,ed,65-74,medium,True,True,True,True,10_analysis_results/visualizations/scenario/ed...


Top 20 Consensus Feature Importance:


,cohort,age_band,bin,feature,combined_importance,shap_norm,ffa_norm
207,ed,75-84,low,pgx_num_drugs,1.0,1.0,1.0
143,ed,65-74,low,pgx_num_drugs,1.0,1.0,1.0
151,ed,65-74,medium,pgx_num_drugs,1.0,1.0,1.0
220,ed,75-84,medium,pgx_num_drugs,1.0,1.0,1.0
243,ed,75-84,extreme,pgx_num_drugs,1.0,1.0,1.0
...,...,...,...,...,...,...,...
242,ed,75-84,high,item_drug_AMOXICILLIN,0.0,0.0,0.0
219,ed,75-84,low,item_drug_ELIQUIS,0.0,0.0,0.0
206,ed,65-74,extreme,item_drug_ALPRAZOLAM,0.0,0.0,0.0
150,ed,65-74,low,item_drug_OMEPRAZOLE,0.0,0.0,0.0


## Model Performance Per Density Bin

### Model Performance Summary

Run last in notebook 3 to confirm output files exist and check the selected model metrics per density bin.

In [12]:
import pandas as pd
from py_helpers.event_density_utils import DENSITY_BINS as _DENSITY_BINS

def _outputs_base():
    for base in (FINAL_MODEL_OUTPUTS, FINAL_MODEL_OUTPUTS_ALT):
        if base and base.exists():
            return base
    return FINAL_MODEL_OUTPUTS

base = _outputs_base()
print("Final model performance - per density bin")
print("=" * 80)
all_metrics = []
for cohort, bands in REQUIRED_COHORTS.items():
    for age_band in bands:
        ab_f = age_band.replace("-", "_")
        for bin_name in _DENSITY_BINS:
            path = (
                base / cohort / ab_f / "bin_models" / bin_name
                / f"{cohort}_{ab_f}_model_metrics_summary.csv"
            )
            if not path.exists():
                continue
            df = pd.read_csv(path)
            selected = df.loc[df["selected"] == True]
            if selected.empty:
                selected = df.head(1)
            for _, row in selected.iterrows():
                all_metrics.append({
                    "cohort": cohort,
                    "age_band": age_band,
                    "bin": bin_name,
                    "model": row["model"],
                    "recall_mean": row["recall_mean"],
                    "pr_auc_mean": row["pr_auc_mean"],
                    "auc_mean": row.get("auc_mean"),
                })
if all_metrics:
    print(pd.DataFrame(all_metrics).to_string(index=False))
else:
    print(f"  No metrics CSVs found under {base}")

Final model performance - per density bin
cohort age_band     bin      model  recall_mean  pr_auc_mean  auc_mean
 falls    65-74     low   Ensemble     0.758644     0.835849  0.827569
 falls    65-74  medium   Ensemble     0.689032     0.891589  0.919105
 falls    65-74    high   Ensemble     0.261818     0.712785  0.918206
 falls    65-74 extreme   Ensemble     0.689032     0.891589  0.919105
 falls    75-84     low   Ensemble     0.831282     0.867491  0.867515
 falls    75-84  medium   Ensemble     0.657647     0.832361  0.901846
 falls    75-84    high   Ensemble     0.028571     0.343226  0.781934
 falls    75-84 extreme   Ensemble     0.657647     0.832361  0.901846
    ed    65-74     low    XGBoost     0.972161     0.925475  0.896069
    ed    65-74  medium XGBoost_RF     0.520000     0.506887  0.873227
    ed    65-74    high XGBoost_RF     0.000000     0.075566  0.676651
    ed    65-74 extreme   CatBoost     0.000000     0.106837  0.510029
    ed    75-84     low    XGBoost 

## Notebook 3 Complete

Run `4_results_review.ipynb` next for the final results review, results artifact sync, and optional EC2 shutdown.

In [ ]:
print("Notebook 3 complete. Run 4_results_review.ipynb for final review, S3 sync, and optional EC2 shutdown.")


In [ ]:
# Spacer intentionally left blank.